# Tribal Fire Response & Capacity Gap Analysis
**Series:** Tribal Fire Science & Indigenous Data Sovereignty  
**Author:** Lilly Jones, PhD  
**Last Updated:** 2025  
**Data Sources:** Census TIGER AIANNH, HIFLD Fire Stations, OpenStreetMap (osmnx)

## Overview
This notebook analyzes fire response capacity and coverage gaps on Tribal lands by mapping
nearby fire stations, calculating network-based response time isochrones, and identifying
Tribal land areas that fall outside reasonable response windows. Fire station locations
come from the HIFLD (Homeland Infrastructure Foundation-Level Data) national dataset.
Road networks are downloaded from OpenStreetMap via osmnx.

## Research Questions
- Which Tribal lands fall outside 30-minute fire response coverage?
- How does coverage vary by agency type (BIA, USFS, BLM, State)?
- Which areas should be prioritized for new stations or mutual aid agreements?

## Outputs
- Interactive and static maps of response coverage
- Network-based isochrones (reachable areas within 10, 20, 30 minutes)
- Coverage gap analysis per Tribal land unit
- Priority ranking for response capacity improvements

## Data Sovereignty Note
> Tribal boundary data comes from Census TIGER AIANNH federal statistical boundaries
> that may not reflect Tribal Nations' own definitions of territory.
> Tribal fire program staffing data is not available from a public API and must be
> supplied directly by or in collaboration with each Tribal Nation.
> See `TRIBAL_FIRE_PROGRAMS` in Section 2 to enter known program data.
> See the acknowledgment below. All geospatial data is real; no synthetic data is used.

In [1]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
from datetime import datetime

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import seaborn as sns
from shapely.geometry import Point
from shapely.ops import unary_union

from src.data import constants, loaders, validators
from src.geo import utils as geo_utils
from src.indigenous.sovereignty import (
    generate_citations,
    print_data_acknowledgment,
)
from src.viz import charts, maps, styles

styles.apply_mpl_style()
%matplotlib inline

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")

print(f"Repo root : {REPO_ROOT}")
print(f"Cache dir : {constants.CACHE_DIR}")
print(f"Output dir: {constants.OUTPUTS_DIR}")
print(f"Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Repo root : C:\Users\gekek\Documents\tfs_refactor
Cache dir : C:\Users\gekek\Documents\tfs_refactor\data\cache
Output dir: C:\Users\gekek\Documents\tfs_refactor\outputs
Analysis run: 2026-03-28 20:10


In [2]:
# Data sovereignty acknowledgment 
print_data_acknowledgment(source_keys=["census_aiannh", "hifld_fire_stations"])

DATA SOVEREIGNTY ACKNOWLEDGMENT
This analysis uses data that describes Indigenous and Tribal lands,
communities, and fire histories. This project is guided by three
complementary data governance frameworks:

OCAP® — Tribal Nations own, control, access, and possess data about
  their own communities and territories.
  Reference: https://fnigc.ca/ocap-training/

CARE  — Data use must deliver Collective Benefit to Indigenous peoples,
  respect their Authority to Control, uphold Responsibility to communities,
  and center Ethics across the full data lifecycle.
  Reference: https://www.gida-global.org/care

FAIR  — Data is Findable, Accessible, Interoperable, and Reusable.
  FAIR governs technical standards; CARE and OCAP® govern the ethical
  obligations to Tribal Nations that FAIR alone does not address.
  Reference: https://www.go-fair.org/fair-principles/

We recognize that:

• Tribal Nations are sovereign governments with the right to control
  data about their own communities and terr

## Configuration

In [3]:
# Analysis configuration 
ANALYSIS_CONFIG = {
    # Response time thresholds (minutes)
    "response_times":       [10, 20, 30],
    "max_response_time":    30,
    # Emergency vehicle speed multiplier over posted limits
    "speed_multiplier":     1.5,
    # Coverage target (%)
    "coverage_target_pct":  90,
}

# Tribal Nations to analyze 
# Names must match Census TIGER 'NAME' field.
# Edit this list to analyze different Tribal Nations.
TRIBES_OF_INTEREST = [
    "Navajo Nation",
    "Fort Apache",
    "San Carlos",
    "Tohono O'odham",
    "White Mountain",
    "Hopi",
]

# Tribal fire program data 
# This information is NOT available from a public API.
# Supply known values here; leave None for unknown.
# Source: BIA Division of Fire Management, direct Tribal contact, or
# published Tribal fire management plans.
TRIBAL_FIRE_PROGRAMS = {
    # "Census NAME": {"has_program": bool, "staff_count": int or None}
    "Navajo Nation":     {"has_program": True,  "staff_count": 45},
    "Fort Apache":       {"has_program": True,  "staff_count": 12},
    "San Carlos":        {"has_program": False, "staff_count": 0},
    "Tohono O'odham":    {"has_program": True,  "staff_count": 8},
    "White Mountain":    {"has_program": True,  "staff_count": None},
    "Hopi":              {"has_program": None,  "staff_count": None},
}

print("Configuration set.")
print(f"  Response time windows : {ANALYSIS_CONFIG['response_times']} minutes")
print(f"  Speed multiplier      : {ANALYSIS_CONFIG['speed_multiplier']}x")
print(f"  Coverage target       : {ANALYSIS_CONFIG['coverage_target_pct']}%")
print(f"  Tribal Nations        : {len(TRIBES_OF_INTEREST)}")

Configuration set.
  Response time windows : [10, 20, 30] minutes
  Speed multiplier      : 1.5x
  Coverage target       : 90%
  Tribal Nations        : 6


## Load Data

In [ ]:
# Tribal land boundaries 
all_tribal = loaders.load_census_aian()
all_tribal = validators.validate_geodataframe(
    all_tribal, "census_aiannh",
    required_columns=["geometry", "NAME"],
)

tribal_lands = all_tribal[
    all_tribal["NAME"].isin(TRIBES_OF_INTEREST)
].copy().reset_index(drop=True)

# Attach fire program data
tribal_lands["has_fire_program"] = tribal_lands["NAME"].map(
    lambda n: TRIBAL_FIRE_PROGRAMS.get(n, {}).get("has_program")
)
tribal_lands["fire_staff_count"] = tribal_lands["NAME"].map(
    lambda n: TRIBAL_FIRE_PROGRAMS.get(n, {}).get("staff_count")
)

if tribal_lands.empty:
    raise ValueError(
        "No Tribal lands matched. Check TRIBES_OF_INTEREST names against "
        "all_tribal['NAME'].unique() for exact spelling."
    )

all_tribal = loaders.load_census_aian()
all_tribal = validators.validate_geodataframe(
    all_tribal, "census_aiannh",
    required_columns=["geometry", "NAME"],
)

tribal_lands = all_tribal[
    all_tribal["NAME"].isin(TRIBES_OF_INTEREST)
].copy().reset_index(drop=True)

# Dissolve duplicate polygons (non-contiguous parcels) by NAME
tribal_lands = (
    tribal_lands
    .dissolve(by="NAME", as_index=False)
    .reset_index(drop=True)
)

# Clip to CONUS — removes spurious Alaska/Hawaii matches
CONUS_BOUNDS = (-127, 24, -65, 50)
conus = geo_utils.bbox_geodataframe(CONUS_BOUNDS)
tribal_lands = tribal_lands[
    tribal_lands.intersects(conus.geometry.iloc[0])
].copy().reset_index(drop=True)

# Attach fire program data
tribal_lands["has_fire_program"] = tribal_lands["NAME"].map(
    lambda n: TRIBAL_FIRE_PROGRAMS.get(n, {}).get("has_program")
)
tribal_lands["fire_staff_count"] = tribal_lands["NAME"].map(
    lambda n: TRIBAL_FIRE_PROGRAMS.get(n, {}).get("staff_count")
)

print(f"Tribal land units loaded: {len(tribal_lands)}")
tribal_lands[["NAME", "has_fire_program", "fire_staff_count"]].to_string(index=False)
print(tribal_lands[["NAME", "has_fire_program", "fire_staff_count"]].to_string(index=False))

Tribal land units loaded: 4
         NAME has_fire_program  fire_staff_count
  Fort Apache             True              12.0
         Hopi             None               NaN
Navajo Nation             True              45.0
   San Carlos            False               0.0
Tribal land units loaded: 4
         NAME has_fire_program  fire_staff_count
  Fort Apache             True              12.0
         Hopi             None               NaN
Navajo Nation             True              45.0
   San Carlos            False               0.0


In [5]:
# Study area bounding box 
# Expand the bounds slightly to capture stations just outside Tribal boundaries
BUFFER_DEG = 0.5
bounds = tribal_lands.total_bounds  # [minx, miny, maxx, maxy]
STUDY_BBOX = (
    bounds[0] - BUFFER_DEG,  # min lon
    bounds[1] - BUFFER_DEG,  # min lat
    bounds[2] + BUFFER_DEG,  # max lon
    bounds[3] + BUFFER_DEG,  # max lat
)
print(f"Study area bbox (lon_min, lat_min, lon_max, lat_max):")
print(f"  {STUDY_BBOX}")

Study area bbox (lon_min, lat_min, lon_max, lat_max):
  (np.float64(-163.959835), np.float64(32.375138228796125), np.float64(-106.44300460496686), np.float64(65.189572))


In [8]:
# Fire stations HIFLD 
# Derives states from the study area; adjust STATE_FILTER manually if needed.
# HIFLD includes federal (BIA, USFS, BLM), state, local, and some Tribal stations.

STATE_FILTER = ["AZ", "NM"]  # Set to states covering your Tribal lands

all_stations = loaders.load_hifld_fire_stations(
    state_filter=STATE_FILTER,
    force_refresh=False,
)
all_stations = validators.validate_geodataframe(
    all_stations, "hifld_fire_stations",
    required_columns=["geometry"],
)

# Clip to study area
bbox_poly = geo_utils.bbox_geodataframe(STUDY_BBOX)
fire_stations = all_stations[
    all_stations.within(bbox_poly.geometry.iloc[0])
].copy().reset_index(drop=True)

print(f"Total HIFLD stations   : {len(all_stations):,}")
print(f"Stations in study area : {len(fire_stations):,}")

if "AGENCYTYPE" in fire_stations.columns:
    print("\nAgency type breakdown:")
    print(fire_stations["AGENCYTYPE"].value_counts().to_string())

fire_stations.head()

HTTPError: 400 Client Error: Bad Request for url: https://services1.arcgis.com/Hp6G80Pky0om7QvQ/arcgis/rest/services/Fire_Stations/FeatureServer/0/query?where=STATE+IN+%28%27AZ%27%2C+%27NM%27%29&outFields=%2A&f=geojson&resultRecordCount=5000

In [ ]:
# Road network OpenStreetMap via osmnx 
# Downloads the drivable road network for the study area.
# First run may take several minutes depending on study area size.
# Results are cached by osmnx automatically.

print("Downloading road network from OpenStreetMap...")
print("(Cached after first download — subsequent runs are fast)")

ox.settings.log_console = False
ox.settings.use_cache  = True

G = ox.graph_from_bbox(
    bbox=(
        STUDY_BBOX[1],  # south (lat min)
        STUDY_BBOX[3],  # north (lat max)
        STUDY_BBOX[0],  # west  (lon min)
        STUDY_BBOX[2],  # east  (lon max)
    ),
    network_type="drive",
    simplify=True,
)
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

print(f"Road network loaded:")
print(f"  Nodes : {len(G.nodes):,}")
print(f"  Edges : {len(G.edges):,}")

## Response Time Isochrones

In [ ]:
def calculate_isochrones(
    G: nx.MultiDiGraph,
    point: Point,
    trip_times: list[int],
    speed_multiplier: float = 1.5,
) -> gpd.GeoDataFrame:
    """
    Calculate network-based isochrones from a fire station location.

    Uses ego_graph to find all nodes reachable within each time window,
    then builds a convex hull around those nodes as the isochrone polygon.
    Emergency vehicle speed is modeled as speed_multiplier × posted limit.

    Parameters
    ----------
    G              : osmnx road network with travel_time edge attribute
    point          : station location (lon, lat)
    trip_times     : list of response time windows in minutes
    speed_multiplier: emergency speed factor over posted limits

    Returns
    -------
    GeoDataFrame with one row per trip_time, columns: geometry, time_minutes
    """
    try:
        center_node = ox.distance.nearest_nodes(G, point.x, point.y)
    except Exception as e:
        print(f"    Could not find nearest node: {e}")
        return gpd.GeoDataFrame()

    # Scale travel times for emergency speed
    G_emerg = G.copy()
    for _, _, _, data in G_emerg.edges(keys=True, data=True):
        if "travel_time" in data:
            data["travel_time"] = data["travel_time"] / speed_multiplier

    polys = []
    for t in sorted(trip_times):
        subgraph = nx.ego_graph(
            G_emerg, center_node,
            radius=t * 60,
            distance="travel_time",
        )
        node_pts = [
            Point(G.nodes[n]["x"], G.nodes[n]["y"])
            for n in subgraph.nodes()
        ]
        if len(node_pts) > 2:
            hull = unary_union([p.buffer(0.005) for p in node_pts]).convex_hull
            polys.append({"geometry": hull, "time_minutes": t})

    return gpd.GeoDataFrame(polys, crs=constants.CRS_GEOGRAPHIC) if polys else gpd.GeoDataFrame()


# Calculate isochrones for each fire station
all_isochrones = []
print(f"Calculating isochrones for {len(fire_stations)} stations...")

for idx, station in fire_stations.iterrows():
    name_col = next(
        (c for c in ["NAME", "FDNAME", "station_name"] if c in station.index), None
    )
    label = station[name_col] if name_col else f"Station {idx}"

    iso = calculate_isochrones(
        G,
        station.geometry,
        ANALYSIS_CONFIG["response_times"],
        ANALYSIS_CONFIG["speed_multiplier"],
    )
    if not iso.empty:
        iso["station_name"] = label
        iso["agency"] = station.get("AGENCYTYPE", "Unknown")
        all_isochrones.append(iso)

if all_isochrones:
    isochrones_gdf = pd.concat(all_isochrones, ignore_index=True)
    isochrones_gdf = gpd.GeoDataFrame(isochrones_gdf, crs=constants.CRS_GEOGRAPHIC)
    print(f"Isochrones calculated: {len(isochrones_gdf):,}")
else:
    isochrones_gdf = gpd.GeoDataFrame()
    print("No isochrones produced — check road network coverage and station locations.")

## Coverage Gap Analysis

In [ ]:
def analyze_coverage_gaps(
    tribal_gdf: gpd.GeoDataFrame,
    isochrones: gpd.GeoDataFrame,
    max_response_time: int = 30,
) -> dict:
    """
    Calculate what percentage of each Tribal land falls within
    max_response_time minutes of a fire station.

    Uses Albers Equal Area for accurate area measurement.
    """
    if isochrones.empty:
        raise ValueError("No isochrone data available for gap analysis.")

    coverage = isochrones[isochrones["time_minutes"] <= max_response_time]
    coverage_union = geo_utils.to_projected(coverage).union_all()

    tribal_proj = geo_utils.to_projected(tribal_gdf)
    results = []

    for _, tribe in tribal_proj.iterrows():
        total_area   = tribe.geometry.area
        covered      = tribe.geometry.intersection(coverage_union)
        covered_area = covered.area if not covered.is_empty else 0
        gap_geom     = tribe.geometry.difference(coverage_union)
        coverage_pct = (covered_area / total_area * 100) if total_area > 0 else 0

        results.append({
            "NAME":              tribe["NAME"],
            "has_fire_program":  tribe.get("has_fire_program"),
            "fire_staff_count":  tribe.get("fire_staff_count"),
            "coverage_pct":      round(coverage_pct, 1),
            "gap_pct":           round(100 - coverage_pct, 1),
            "geometry":          tribe.geometry,
            "gap_geometry":      gap_geom if not gap_geom.is_empty else None,
        })

    result_gdf = gpd.GeoDataFrame(results, crs="EPSG:5070").to_crs(
        constants.CRS_GEOGRAPHIC
    )

    summary = {
        "total_tribal_units":    len(result_gdf),
        "fully_covered":         int((result_gdf["coverage_pct"] >= 99).sum()),
        "partially_covered":     int(((result_gdf["coverage_pct"] > 0) & (result_gdf["coverage_pct"] < 99)).sum()),
        "no_coverage":           int((result_gdf["coverage_pct"] == 0).sum()),
        "avg_coverage_pct":      round(result_gdf["coverage_pct"].mean(), 1),
        "with_fire_program":     int((result_gdf["has_fire_program"] == True).sum()),
        "without_fire_program":  int((result_gdf["has_fire_program"] == False).sum()),
        "program_data_missing":  int(result_gdf["has_fire_program"].isna().sum()),
    }

    return {"coverage": result_gdf, "summary": summary}


gap_analysis = analyze_coverage_gaps(
    tribal_lands,
    isochrones_gdf,
    max_response_time=ANALYSIS_CONFIG["max_response_time"],
)

print("COVERAGE GAP ANALYSIS")
print("=" * 50)
for k, v in gap_analysis["summary"].items():
    print(f"  {k.replace('_', ' ').title():<28}: {v}")

print("\nCoverage by Tribal Land:")
print(
    gap_analysis["coverage"][["NAME", "coverage_pct", "gap_pct", "has_fire_program", "fire_staff_count"]]
    .sort_values("coverage_pct")
    .to_string(index=False)
)

## Priority Ranking

In [ ]:
def prioritize_gaps(coverage_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Score each Tribal land unit by urgency of response capacity improvement.

    Scoring (100 points max)
    ------------------------
    Coverage gap   : 0–40 pts  (proportional to % uncovered)
    No fire program: 30 pts    (binary)
    Low staffing   : 0–30 pts  (relative to max known staff count)

    Missing program or staffing data is treated conservatively
    (i.e. unknown = possible gap, partial points assigned).
    """
    df = coverage_gdf.copy()
    df["priority_score"] = 0.0

    # Coverage gap (0–40)
    df["priority_score"] += df["gap_pct"] * 0.4

    # No fire program (30); unknown gets 15
    df["priority_score"] += df["has_fire_program"].map(
        {False: 30, True: 0, None: 15}
    ).fillna(15)

    # Low staffing (0–30)
    max_staff = df["fire_staff_count"].max()
    if pd.notna(max_staff) and max_staff > 0:
        df["staffing_score"] = df["fire_staff_count"].apply(
            lambda x: (1 - x / max_staff) * 30 if pd.notna(x) else 15
        )
    else:
        df["staffing_score"] = 15
    df["priority_score"] += df["staffing_score"]

    df["priority_category"] = pd.cut(
        df["priority_score"],
        bins=[0, 35, 65, 100],
        labels=["Low", "Medium", "High"],
    )

    return df.sort_values("priority_score", ascending=False)


priority_df = prioritize_gaps(gap_analysis["coverage"])

print("PRIORITY RANKING")
print("=" * 70)
print(
    priority_df[[
        "NAME", "coverage_pct", "has_fire_program",
        "fire_staff_count", "priority_score", "priority_category",
    ]].to_string(index=False)
)
print(f"\nHigh priority   : {(priority_df['priority_category'] == 'High').sum()}")
print(f"Medium priority : {(priority_df['priority_category'] == 'Medium').sum()}")
print(f"Low priority    : {(priority_df['priority_category'] == 'Low').sum()}")

## Visualizations

In [ ]:
# Coverage map 
import contextily as ctx

fig, ax = plt.subplots(figsize=(14, 10))

# Reproject to Web Mercator for basemap
tribal_3857   = tribal_lands.to_crs(epsg=3857)
stations_3857 = fire_stations.to_crs(epsg=3857)

# Tribal lands
tribal_3857.plot(
    ax=ax, color=styles.EARTH_BROWN, edgecolor="black",
    alpha=0.3, linewidth=1, label="Tribal Lands",
)

# Isochrones — green → yellow → orange by response time
iso_colors = {10: "#27AE60", 20: "#F39C12", 30: "#E67E22"}
if not isochrones_gdf.empty:
    for t in sorted(isochrones_gdf["time_minutes"].unique(), reverse=True):
        iso_sub = isochrones_gdf[isochrones_gdf["time_minutes"] == t].to_crs(epsg=3857)
        iso_sub.plot(
            ax=ax, color=iso_colors.get(t, "gray"),
            alpha=0.18, edgecolor="none",
            label=f"{t}-min response",
        )

# Fire stations
stations_3857.plot(
    ax=ax, color=styles.EMBER_RED, marker="*",
    markersize=120, edgecolor="black", linewidth=0.5,
    label="Fire Stations", zorder=5,
)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_axis_off()
ax.set_title(
    f"Fire Response Coverage for Tribal Lands\n"
    f"(Emergency vehicle speed: {ANALYSIS_CONFIG['speed_multiplier']}x posted limit)",
    fontsize=13, fontweight="bold",
)
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
charts.save_figure(fig, "outputs/figures/tribal_fire_coverage_map.png")
plt.show()

In [ ]:
# Coverage and priority bar charts 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Chart 1: Coverage % by Tribal land
cov = gap_analysis["coverage"].sort_values("coverage_pct")
bar_colors = [
    styles.SAGE_GREEN if v >= ANALYSIS_CONFIG["coverage_target_pct"]
    else styles.FIRE_ORANGE if v >= 50
    else styles.EMBER_RED
    for v in cov["coverage_pct"]
]
ax1.barh(cov["NAME"], cov["coverage_pct"], color=bar_colors, alpha=0.85)
ax1.axvline(
    ANALYSIS_CONFIG["coverage_target_pct"],
    color=styles.CHARCOAL, linestyle="--", linewidth=1.5,
    label=f"{ANALYSIS_CONFIG['coverage_target_pct']}% target",
)
ax1.set_xlabel("Coverage (%)")
ax1.set_title(
    f"Fire Response Coverage\n({ANALYSIS_CONFIG['max_response_time']}-min window)",
    fontsize=12, fontweight="bold",
)
ax1.legend(fontsize=9)
ax1.set_xlim(0, 105)
sns.despine(ax=ax1)

# Chart 2: Priority scores
p_colors = {
    "High": styles.EMBER_RED,
    "Medium": styles.FIRE_ORANGE,
    "Low": styles.SAGE_GREEN,
}
bar_p_colors = [p_colors.get(str(c), styles.SMOKE_GRAY) for c in priority_df["priority_category"]]
ax2.barh(priority_df["NAME"], priority_df["priority_score"],
         color=bar_p_colors, alpha=0.85)
ax2.set_xlabel("Priority Score")
ax2.set_title("Response Capacity Priority Score",
              fontsize=12, fontweight="bold")
from matplotlib.patches import Patch
ax2.legend(
    handles=[Patch(color=v, label=k) for k, v in p_colors.items()],
    fontsize=9,
)
sns.despine(ax=ax2)

plt.suptitle("Tribal Fire Response Capacity Analysis",
             fontsize=14, fontweight="bold")
plt.tight_layout()
charts.save_figure(fig, "outputs/figures/tribal_fire_capacity_charts.png")
plt.show()

## Recommendations

In [ ]:
def generate_recommendations(
    coverage_gdf: gpd.GeoDataFrame,
    priority_gdf: gpd.GeoDataFrame,
    coverage_target: float = 90,
) -> list[dict]:
    """Generate prioritized, evidence-based recommendations."""
    recs = []

    # High priority, no fire program
    critical = priority_gdf[
        (priority_gdf["priority_category"] == "High") &
        (priority_gdf["has_fire_program"] == False)
    ]
    if not critical.empty:
        recs.append({
            "priority": "CRITICAL",
            "category": "Tribal Fire Program Development",
            "description": (
                f"{len(critical)} high-priority Tribal land unit(s) lack a fire program "
                "and have significant coverage gaps. Federal support for program "
                "establishment should be prioritized under BIA Division of Fire Management."
            ),
            "affected": critical["NAME"].tolist(),
        })

    # Poor coverage regardless of program
    poor = coverage_gdf[coverage_gdf["coverage_pct"] < 50]
    if not poor.empty:
        recs.append({
            "priority": "HIGH",
            "category": "Station Placement / Mutual Aid Agreements",
            "description": (
                f"{len(poor)} unit(s) have less than 50% coverage within "
                f"{ANALYSIS_CONFIG['max_response_time']} minutes. "
                "New station siting analysis or pre-positioned resources "
                "during high-risk periods should be evaluated."
            ),
            "affected": poor["NAME"].tolist(),
        })

    # Below coverage target
    below_target = coverage_gdf[
        (coverage_gdf["coverage_pct"] < coverage_target) &
        (coverage_gdf["coverage_pct"] >= 50)
    ]
    if not below_target.empty:
        recs.append({
            "priority": "MEDIUM",
            "category": f"Coverage Enhancement (below {coverage_target}% target)",
            "description": (
                f"{len(below_target)} unit(s) are between 50–{coverage_target}% covered. "
                "Road network improvements, pre-positioning, or expanded mutual aid "
                "agreements would bring these units to target."
            ),
            "affected": below_target["NAME"].tolist(),
        })

    # Missing program data
    unknown = coverage_gdf[coverage_gdf["has_fire_program"].isna()]
    if not unknown.empty:
        recs.append({
            "priority": "DATA GAP",
            "category": "Tribal Fire Program Data Collection",
            "description": (
                f"Fire program status is unknown for {len(unknown)} unit(s). "
                "Direct engagement with Tribal fire managers is needed to "
                "complete this assessment. Update TRIBAL_FIRE_PROGRAMS in Section 1."
            ),
            "affected": unknown["NAME"].tolist(),
        })

    return recs


recommendations = generate_recommendations(
    gap_analysis["coverage"],
    priority_df,
    coverage_target=ANALYSIS_CONFIG["coverage_target_pct"],
)

print("RECOMMENDATIONS")
print("=" * 70)
for i, rec in enumerate(recommendations, 1):
    print(f"\n{i}. [{rec['priority']}] {rec['category']}")
    print(f"   {rec['description']}")
    print(f"   Affected: {', '.join(rec['affected'])}")

## Exports

In [ ]:
# Export spatial outputs 
geo_exports = {
    "tribal_fire_coverage.geojson":  gap_analysis["coverage"][["NAME", "has_fire_program", "fire_staff_count", "coverage_pct", "gap_pct", "geometry"]],
    "fire_stations_study_area.geojson": fire_stations,
}
if not isochrones_gdf.empty:
    geo_exports["response_time_isochrones.geojson"] = isochrones_gdf

for fname, gdf in geo_exports.items():
    path = constants.OUTPUTS_DIR / fname
    gdf.to_file(path, driver="GeoJSON")
    print(f"Exported → {path.relative_to(REPO_ROOT)}")

# Export tabular outputs 
csv_exports = {
    "tribal_fire_capacity_priority.csv": priority_df[[
        "NAME", "coverage_pct", "gap_pct", "has_fire_program",
        "fire_staff_count", "priority_score", "priority_category",
    ]],
    "tribal_fire_recommendations.csv": pd.DataFrame(recommendations),
}
for fname, df in csv_exports.items():
    path = constants.OUTPUTS_DIR / fname
    df.to_csv(path, index=False)
    print(f"Exported → {path.relative_to(REPO_ROOT)}")

## Summary & Findings

*(Fill in after running with your study area data.)*

Key questions to address in the narrative:
- Which Tribal lands have the most significant response coverage gaps?
- Does the presence of a Tribal fire program correlate with better coverage,
  or are some programs present but still underserved by external station proximity?
- What are the limitations of this analysis? OSM road networks may be incomplete
  in rural Tribal areas. HIFLD station data may not include all Tribal-operated
  stations. Isochrones assume road travel only — helicopter response is not modeled.
- What would direct engagement with Tribal fire managers add to this picture?
  This notebook provides a geospatial baseline, not a substitute for
  Tribal-led capacity assessment.

---
## References

In [ ]:
print(generate_citations(["census_aiannh", "hifld_fire_stations"]))